# Rasa Demo Bot Analytics Walkthrough

This notebook demonstrates how to generate simulated Rasa conversation logs, analyse fallback events, and build dashboard-ready visualisations. It is designed to be run in Google Colab and relies only on open-source libraries available via `pip`.

In [ ]:
# If running in a fresh Google Colab session, install the required packages
%pip install --quiet pandas numpy matplotlib seaborn plotly scikit-learn scipy pyyaml


In [ ]:
import json
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.cluster import DBSCAN
import yaml


## 1. Simulate Rasa Conversation Logs
The helper below creates a realistic-looking log dataset compatible with Rasa tracker exports. Each row represents a user message or bot action annotated with metadata that can be aggregated for analytics.

In [ ]:
def simulate_rasa_logs(num_sessions: int = 150, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    base_time = datetime.now() - timedelta(days=7)
    intents = [
        'greet', 'identify_leaf_spot', 'detect_powdery_mildew', 'upload_leaf_photo',
        'record_voice_note', 'provide_symptom_audio', 'ask_followup_question',
        'request_treatment_plan', 'affirm', 'goodbye', 'out_of_scope', 'disease_photo_unclear'
    ]
    channels = ['webchat', 'whatsapp', 'field_app', 'voice_line']
    modality_by_intent = {
        'upload_leaf_photo': 'image',
        'detect_powdery_mildew': 'image',
        'identify_leaf_spot': 'image',
        'record_voice_note': 'audio',
        'provide_symptom_audio': 'audio',
        'ask_followup_question': 'text',
        'request_treatment_plan': 'text',
        'disease_photo_unclear': 'image',
        'affirm': 'text',
        'goodbye': 'text',
        'out_of_scope': 'text',
        'greet': 'text'
    }
    steps = []
    for session_id in range(1, num_sessions + 1):
        start_time = base_time + timedelta(minutes=int(rng.uniform(0, 60 * 24 * 7)))
        num_turns = rng.integers(5, 14)
        channel = rng.choice(channels, p=[0.45, 0.25, 0.2, 0.1])
        fallback_probability = 0.06 + 0.18 * (channel == 'webchat') + 0.05 * (channel == 'voice_line')
        journey_state = 'in_progress'
        for turn in range(num_turns):
            timestamp = start_time + timedelta(minutes=turn * rng.uniform(0.5, 3.0))
            intent = rng.choice(intents, p=[0.1, 0.12, 0.08, 0.12, 0.05, 0.06, 0.1, 0.1, 0.06, 0.08, 0.07, 0.06])
            modality = modality_by_intent.get(intent, 'text')
            is_fallback = rng.random() < fallback_probability if intent in {'out_of_scope', 'disease_photo_unclear'} else False
            model_confidence = float(rng.uniform(0.55, 0.95)) if modality != 'text' else float(rng.uniform(0.6, 0.99))
            if is_fallback:
                journey_state = 'fallback'
            elif intent == 'goodbye':
                journey_state = 'resolved'
            steps.append({
                'session_id': f'S{session_id:04d}',
                'timestamp': timestamp,
                'intent': intent,
                'channel': channel,
                'modality': modality,
                'model_confidence': model_confidence,
                'is_fallback': is_fallback,
                'journey_state': journey_state
            })
    df = pd.DataFrame(steps).sort_values('timestamp').reset_index(drop=True)
    return df

logs_df = simulate_rasa_logs()
logs_df.head()


## 2. Fallback Optimisation Analytics
We focus on fallback analysis to improve recovery flows. The metric suite below tracks fallback frequency by channel and identifies high-risk intents.

In [ ]:
channel_group = logs_df.groupby('channel')['is_fallback'].agg(['sum', 'count']).reset_index()
channel_group['fallback_rate'] = channel_group['sum'] / channel_group['count']
z = stats.norm.ppf(0.975)
channel_group['wilson_lower'] = (
    (channel_group['fallback_rate'] + z**2 / (2 * channel_group['count']) -
     z * np.sqrt((channel_group['fallback_rate'] * (1 - channel_group['fallback_rate']) + z**2 / (4 * channel_group['count'])) / channel_group['count'])) /
    (1 + z**2 / channel_group['count'])
)
channel_group['wilson_upper'] = (
    (channel_group['fallback_rate'] + z**2 / (2 * channel_group['count']) +
     z * np.sqrt((channel_group['fallback_rate'] * (1 - channel_group['fallback_rate']) + z**2 / (4 * channel_group['count'])) / channel_group['count'])) /
    (1 + z**2 / channel_group['count'])
)
fallback_summary = channel_group[['channel', 'fallback_rate', 'wilson_lower', 'wilson_upper']].round(3)
fallback_summary


In [ ]:
fig = px.bar(
    fallback_summary,
    x="channel",
    y="fallback_rate",
    title="Fallback Rate by Channel",
    text_auto=".1%"
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

In [ ]:
intent_fallback = (
    logs_df.groupby("intent")["is_fallback"].mean().rename("fallback_rate")
    .reset_index()
)
intent_fallback.sort_values("fallback_rate", ascending=False).head(10)

### Session-Level Heatmap
This heatmap helps identify when fallback events occur in the conversation timeline. The darker the colour, the later the fallback.

In [ ]:
session_pivots = (
    logs_df.assign(step=lambda df: df.groupby("session_id").cumcount())
    .pivot_table(
        index="session_id",
        columns="step",
        values="is_fallback",
        aggfunc="max",
        fill_value=0
    )
)
plt.figure(figsize=(10, 6))
sns.heatmap(session_pivots.iloc[:40], cmap="YlOrRd", cbar_kws={"label": "Fallback Occurrence"})
plt.title("Fallback Occurrence Heatmap (first 40 sessions)")
plt.xlabel("Turn Index")
plt.ylabel("Session")
plt.tight_layout()
plt.show()

## 3. Dashboard-Ready Aggregations
The following cells derive metrics for a chatbot performance dashboard covering platform performance, user journey attribution, and feedback signals.

In [ ]:
top_intents = (
    logs_df.groupby("intent").size().sort_values(ascending=False).reset_index(name="count")
)
px.bar(top_intents, x="intent", y="count", title="Top User Intents").show()

In [ ]:
journey_summary = (
    logs_df.groupby(["session_id"]).agg(
        channel=("channel", "first"),
        resolved=("journey_state", lambda s: (s == "resolved").any()),
        fallback=("is_fallback", "any")
    )
)
journey_counts = {
    "Start": len(journey_summary),
    "Resolution": int(journey_summary["resolved"].sum()),
    "Fallback Exit": int((~journey_summary["resolved"] & journey_summary["fallback"]).sum())
}
funnel_fig = go.Figure(go.Funnel(
    y=list(journey_counts.keys()),
    x=list(journey_counts.values()),
    textinfo="value+percent initial"
))
funnel_fig.update_layout(title="User Journey Funnel")
funnel_fig.show()

In [ ]:
feedback_counts = pd.DataFrame({
    "outcome": ["Successful", "Fallback"],
    "count": [journey_counts["Resolution"], journey_counts["Fallback Exit"]]
})
px.pie(feedback_counts, names="outcome", values="count", title="Conversation Outcomes", hole=0.3).show()

In [ ]:
sessions_over_time = (
    logs_df.groupby(pd.Grouper(key="timestamp", freq="6H"))["session_id"].nunique().reset_index(name="sessions")
)
px.line(sessions_over_time, x="timestamp", y="sessions", title="Sessions over Time").show()

## 2a. Extended Experimentation: A/B Test Simulation

We simulate two response variants—concise vs. explanatory guidance—assigned at the session level. The Welch t-test compares resolution rates and post-conversation satisfaction scores.


In [ ]:
rng = np.random.default_rng(7)
variant_assignments = pd.DataFrame({
    'session_id': logs_df['session_id'].unique(),
    'variant': rng.choice(['concise', 'detailed'], size=logs_df['session_id'].nunique())
})

variant_outcomes = variant_assignments.merge(
    journey_summary[['resolved']], left_on='session_id', right_index=True, how='left'
).fillna({'resolved': False})
variant_outcomes['satisfaction'] = rng.normal(loc=4.1, scale=0.6, size=len(variant_outcomes))
variant_outcomes.loc[variant_outcomes['variant'] == 'concise', 'satisfaction'] -= 0.2

ab_resolution = variant_outcomes.groupby('variant')['resolved'].mean().reset_index(name='resolution_rate')
concise_scores = variant_outcomes.loc[variant_outcomes['variant'] == 'concise', 'satisfaction']
detailed_scores = variant_outcomes.loc[variant_outcomes['variant'] == 'detailed', 'satisfaction']
welch_test = stats.ttest_ind(concise_scores, detailed_scores, equal_var=False)

ab_test_results = {
    'resolution_rates': ab_resolution.to_dict(orient='records'),
    'satisfaction_mean_concise': float(concise_scores.mean()),
    'satisfaction_mean_detailed': float(detailed_scores.mean()),
    'welch_t_statistic': float(welch_test.statistic),
    'welch_p_value': float(welch_test.pvalue)
}
ab_test_results


## 2b. Statistical Quality Gate: Fallback Rate Comparison

We estimate whether the observed decrease in fallback rate after a retraining event is statistically significant using a two-proportion z-test.


In [ ]:
pre_retrain = logs_df.sample(frac=0.5, random_state=21)
post_retrain = logs_df.drop(pre_retrain.index)

pre_rate = pre_retrain['is_fallback'].mean()
post_rate = post_retrain['is_fallback'].mean()

n1, n2 = len(pre_retrain), len(post_retrain)

p_pool = (pre_retrain['is_fallback'].sum() + post_retrain['is_fallback'].sum()) / (n1 + n2)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
z_score = (post_rate - pre_rate) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))

fallback_rate_test = {
    'pre_rate': float(pre_rate),
    'post_rate': float(post_rate),
    'z_score': float(z_score),
    'p_value': float(p_value)
}
fallback_rate_test


## 2c. Intent Drift and Dialogue Anomaly Detection

Low-density clusters in the embedding space highlight sessions that may require new intents or human review.


In [ ]:
rng = np.random.default_rng(29)
embedding_dim = 16
unique_sessions = logs_df['session_id'].unique()
embeddings = rng.normal(size=(len(unique_sessions), embedding_dim))

clustering = DBSCAN(eps=1.6, min_samples=5).fit(embeddings)
anomaly_flags = pd.DataFrame({
    'session_id': unique_sessions,
    'cluster': clustering.labels_
})
anomaly_flags['is_anomaly'] = anomaly_flags['cluster'] == -1

anomaly_summary = anomaly_flags.groupby('is_anomaly').size().reset_index(name='count')
anomaly_examples = logs_df[logs_df['session_id'].isin(anomaly_flags.loc[anomaly_flags['is_anomaly'], 'session_id'])]

{
    'anomaly_counts': anomaly_summary.to_dict(orient='records'),
    'sample_sessions': anomaly_examples[['session_id', 'intent', 'modality']].head(10).to_dict(orient='records')
}


## 3a. Dashboard Configuration Export

To streamline sharing with BI or product teams, we serialise the dashboard layout and associated dataset references as a YAML configuration.


In [ ]:
dashboard_config = {
    'panels': [
        {'title': 'Fallback Rate by Channel', 'type': 'bar', 'dataset': 'fallback_summary'},
        {'title': 'Top User Intents', 'type': 'bar', 'dataset': 'top_intents'},
        {'title': 'User Journey Funnel', 'type': 'funnel', 'dataset': 'journey_counts'},
        {'title': 'Conversation Outcomes', 'type': 'pie', 'dataset': 'feedback_counts'},
        {'title': 'Sessions over Time', 'type': 'line', 'dataset': 'sessions_over_time'}
    ],
    'filters': ['channel', 'date_range'],
    'last_updated': datetime.utcnow().isoformat()
}

with open('plantguard_dashboard_config.yaml', 'w') as fh:
    yaml.safe_dump(dashboard_config, fh)

plantguard_dashboard_config = dashboard_config
dashboard_config


## 4. Export Aggregated Metrics
These tables can be sent to downstream BI tools or dashboards.

In [ ]:
metrics_bundle = {
    'fallback_summary': fallback_summary.to_dict(orient='records'),
    'intent_counts': top_intents.to_dict(orient='records'),
    'journey_counts': journey_counts,
    'ab_test_results': ab_test_results,
    'fallback_rate_test': fallback_rate_test,
    'anomaly_counts': anomaly_summary.to_dict(orient='records')
}
with open('rasa_demo_metrics.json', 'w') as fp:
    json.dump(metrics_bundle, fp, indent=2)
metrics_bundle
